In [30]:
from azureml.core import Workspace, Dataset

from azureml.datadrift import DataDriftDetector

from datetime import datetime, timedelta

In [31]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

In [32]:
dstore.upload('data', 'florida-weather-2019', overwrite=True)

Uploading an estimated of 10 files
Uploading data/2019/01/data.parquet
Uploading data/2019/02/data.parquet
Uploading data/2019/03/data.parquet
Uploading data/2019/04/data.parquet
Uploading data/2019/05/data.parquet
Uploading data/2019/06/data.parquet
Uploading data/2019/07/data.parquet
Uploading data/2019/08/data.parquet
Uploading data/2019/09/data.parquet
Uploading data/2019/10/data.parquet
Uploaded data/2019/10/data.parquet, 1 files out of an estimated total of 10
Uploaded data/2019/01/data.parquet, 2 files out of an estimated total of 10
Uploaded data/2019/05/data.parquet, 3 files out of an estimated total of 10
Uploaded data/2019/04/data.parquet, 4 files out of an estimated total of 10
Uploaded data/2019/09/data.parquet, 5 files out of an estimated total of 10
Uploaded data/2019/06/data.parquet, 6 files out of an estimated total of 10
Uploaded data/2019/03/data.parquet, 7 files out of an estimated total of 10
Uploaded data/2019/07/data.parquet, 8 files out of an estimated total of 

$AZUREML_DATAREFERENCE_07951ab3398f45858e32858a860d89d4

In [33]:
dset   = Dataset.Tabular.from_parquet_files(dstore.path('florida-weather-2019/**/data.parquet'))

In [34]:
target = dset.with_timestamp_columns('datetime')
target = target.register(ws, 'target')

In [35]:
target = Dataset.get_by_name(ws, 'target')

In [36]:
baseline = target.time_before(datetime(2019, 2, 1))

In [37]:
features = ['latitude', 'longitude', 'elevation', 'windAngle', 'windSpeed', 'temperature', 'snowDepth', 'stationName', 'countryOrRegion']

In [40]:
monitor = DataDriftDetector.create_from_datasets(ws, 'drift_monitor2', baseline, target, 
                                                      compute_target_name='cpu-cluster', 
                                                      frequency='Week', 
                                                      feature_list=None, 
                                                      drift_threshold=None, 
                                                      latency=0)

2019-10-07 19:57:29,623 - azureml.datadrift._logging._telemetry_logger.datadriftdetector.create_from_datasets - ERROR - ActivityCompleted: Activity=datadriftdetector.create_from_datasets, HowEnded=Failure, Duration=5239.62 [ms], Exception=UserErrorException
Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/_restclient/clientbase.py", line 287, in _execute_func_internal
    return func(*args, **kwargs)
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/datadrift/_restclient/rest_client.py", line 319, in get_by_name
    raise HttpOperationError(self._deserialize, response)
msrest.exceptions.HttpOperationError: Operation returned an invalid status code 'Forbidden'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/telemetry/activity.py", line 98, in log_activity
    yield activi

UserErrorException: UserErrorException:
	Message: 
Operation returned an invalid status code 'Forbidden'. The possible reason could be:
1. You are not authorized to access this resource, or directory listing denied.
2. you may not login your azure service, or use other subscription, you can check your
default account by running azure cli commend:
'az account list -o table'.
3. You have multiple objects/login session opened, please close all session and try again.
                
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "\nOperation returned an invalid status code 'Forbidden'. The possible reason could be:\n1. You are not authorized to access this resource, or directory listing denied.\n2. you may not login your azure service, or use other subscription, you can check your\ndefault account by running azure cli commend:\n'az account list -o table'.\n3. You have multiple objects/login session opened, please close all session and

In [ ]:
monitor = DataDriftDetector.get_by_name(ws, 'drift_monitor2')

monitor.update(feature_list=None)

In [ ]:
backfill1 = monitor.backfill(datetime(2019, 1, 1), datetime(2019, 5, 1), feature_list=features)
backfill1

In [ ]:
backfill2 = monitor.backfill(datetime(2019, 5, 1), datetime.today(), feature_list=features)
backfill2

In [ ]:
help(DataDriftDetector)